In [ ]:
# ===============================
# Reproducible MobileNetV2 Training Script
# ===============================

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# -------------------------------
# 1. Reproducibility
# -------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------------
# 2. Dataset Path
# -------------------------------
DATA_DIR = "BUSI_dataset"  # Place dataset here

IMG_SIZE = 192
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 150

# -------------------------------
# 3. Load Dataset
# -------------------------------
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

class_names = dataset.class_names
num_classes = len(class_names)

# -------------------------------
# 4. Train / Validation Split (80/20)
# -------------------------------
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# -------------------------------
# 5. Build Model
# -------------------------------
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# -------------------------------
# 6. Early Stopping
# -------------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

# -------------------------------
# 7. Train Model
# -------------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop]
)

# -------------------------------
# 8. Evaluate
# -------------------------------
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"Validation Accuracy: {val_accuracy:.4f}")

# -------------------------------
# 9. Confusion Matrix
# -------------------------------
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# -------------------------------
# 10. Save Model
# -------------------------------
model.save("best_model.h5")
print("Model saved as best_model.h5")